# RSNA Knee — Engine 12 & 13 ALL FOLDS (1-4) OOF Prediction Extraction

**Purpose:** Extract Out-of-Fold predictions for all 4 folds of both engines.
- Engine 12: ConvNeXt-Large-384 Gated MIL (Folds 1,2,3,4)
- Engine 13: Swin-Base-384 Asymmetric Focal MIL (Folds 1,2,3,4)
- Outputs: Combined OOF parquet + per-fold CSVs pushed to Kaggle
- Run on A100 GPU (Colab Pro) — ~30 min total

In [ ]:
# 1. Environment & GPU Diagnostic
!nvidia-smi
!pip install -q timm pyarrow kaggle pandas numpy torch torchvision scikit-learn --no-cache-dir
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')

In [ ]:
# 2. Authenticate Kaggle & Download Assets
import os, json
os.makedirs('/root/.kaggle', exist_ok=True)
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_bf1b2382f2c6c8823e02df74217ba8e2'
os.environ['KAGGLE_USERNAME'] = 'cyruspinto'
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': 'cyruspinto', 'key': 'KGAT_bf1b2382f2c6c8823e02df74217ba8e2'}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle Authenticated as cyruspinto!')

os.makedirs('/content/corpus', exist_ok=True)
os.makedirs('/content/labels', exist_ok=True)
os.makedirs('/content/rsna_comp', exist_ok=True)

!kaggle datasets download -d dreaddevelopment/knee-raptor-corpus -p /content/corpus --unzip -q
!kaggle datasets download -d dreaddevelopment/knee-raptor-corpus-ext -p /content/corpus --unzip -q
!kaggle datasets download -d cyruspinto/rsna-knee-super-consensus-labels-0916 -p /content/labels --unzip -q
!kaggle competitions download -c rsna-knee-abnormality-detection -f train.csv -p /content/rsna_comp -q
!unzip -q -o /content/rsna_comp/train.csv.zip -d /content/rsna_comp/ 2>/dev/null || true
print('All training corpora downloaded and verified!')

In [ ]:
# 3. Generate Fold Indices (Seed=42, 5-Fold Stratified by Patient)
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold

TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

train_df = pd.read_csv('/content/rsna_comp/train.csv')
print('Total studies:', len(train_df))

# Group by StudyInstanceUID and get max label per target
group_labels = train_df.groupby('StudyInstanceUID')[TARGETS].max().reset_index()
print('Unique studies:', len(group_labels))

# 5-Fold split
kf = KFold(n_splits=5, shuffle=True, random_state=42)
group_labels['fold'] = -1
for fold, (_, val_idx) in enumerate(kf.split(group_labels)):
    group_labels.loc[val_idx, 'fold'] = fold

# Folds 1-4 are our validation folds (fold 0 was used for earlier experiments)
fold_studies = {}
for fold in [1, 2, 3, 4]:
    fold_studies[fold] = group_labels[group_labels['fold'] == fold]['StudyInstanceUID'].values
    print(f'Fold {fold}: {len(fold_studies[fold])} studies')

In [ ]:
# 4. Load Labels & Corpus Data
import glob

# Load consensus labels
label_files = glob.glob('/content/labels/*.csv')
if not label_files:
    label_files = glob.glob('/content/rsna-knee-super-consensus-labels-0916/*.csv')
labels_df = pd.concat([pd.read_csv(f) for f in label_files], ignore_index=True)
labels_df = labels_df.drop_duplicates(subset=['StudyInstanceUID'])
print('Labels loaded:', len(labels_df))

# Load corpus
vol_files = glob.glob('/content/corpus/*.npy')
if len(vol_files) == 1:
    vols = np.load(vol_files[0], mmap_mode='r')
else:
    parts = [np.load(f, mmap_mode='r') for f in sorted(vol_files)]
    vols = np.concatenate(parts, axis=0)
print('Volumes shape:', vols.shape)

masks = np.load('/content/corpus/masks.npy', mmap_mode='r')
print('Masks shape:', masks.shape)

# Build StudyInstanceUID -> row index
id2row = {}
with open('/content/corpus/study_ids.txt', 'r') as f:
    for i, line in enumerate(f):
        uid = line.strip()
        if uid:
            id2row[uid] = i
print('Study ID map:', len(id2row))

In [ ]:
# 5. Build Study Windows Dataset (replicates training logic)
import random
from torch.utils.data import Dataset, DataLoader

class TwoPartVols:
    def __init__(self, a, b):
        self.a, self.b = a, b
        self.n_a = a.shape[0]
        self.shape = (a.shape[0] + (b.shape[0] if b is not None else 0),) + tuple(a.shape[1:])
    def __len__(self): return self.shape[0]
    def __getitem__(self, row):
        return self.a[row] if row < self.n_a else self.b[row - self.n_a]

class StudyWindowsDataset(Dataset):
    def __init__(self, ids, id2row, labels, vols, masks, res=384, k=16, train=True):
        self.ids = ids
        self.id2row = id2row
        self.labels = labels
        self.vols = vols
        self.masks = masks
        self.res = res
        self.k = k
        self.train = train

    def __len__(self): return len(self.ids)

    def _centers(self, valid, count, d_len):
        lo, hi = int(valid.min()), int(valid.max())
        cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
        if not cs:
            cs = [max(1, min((lo + hi) // 2, d_len - 2))]
        if self.train:
            reps = count // len(cs) + 1
            pool = (cs * reps)
            random.shuffle(pool)
            return sorted(pool[:count])
        idx = np.linspace(0, len(cs) - 1, count).round().astype(int)
        return sorted([cs[i] for i in idx])

    def __getitem__(self, i):
        uid = self.ids[i]
        row = self.id2row[uid]
        vol = self.vols[row]
        mask = self.masks[row]
        y = self.labels.loc[uid].values.astype(np.float32)
        valid = np.where(mask == 1)[0]
        centers = self._centers(valid, self.k, vol.shape[0])
        half = self.res // 2
        windows = []
        for c in centers:
            lo = max(0, c - half)
            hi = lo + self.res
            if hi > vol.shape[0]:
                hi = vol.shape[0]
                lo = hi - self.res
            w = vol[lo:hi]
            m = mask[lo:hi]
            windows.append((w, m))
        # Stack windows: (K, C, H, W)
        imgs = torch.stack([torch.from_numpy(w).float().permute(2, 0, 1) for w, _ in windows])
        masks_t = torch.stack([torch.from_numpy(m).float() for _, m in windows])
        return imgs, masks_t, y, uid

In [ ]:
# 6. Engine 12: ConvNeXt-Large-384 Inference (All 4 Folds)
import torch, torch.nn as nn, torch.nn.functional as F
import timm
from tqdm import tqdm

class ConvNeXtMILClassifier(nn.Module):
    def __init__(self, backbone, feat_dim=1536, num_targets=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.feat_dim = feat_dim
        self.norm = nn.LayerNorm(feat_dim)
        self.gate = nn.Sequential(
            nn.Tanh(),
            nn.Dropout(drop),
            nn.Linear(256, num_targets),
            nn.Sigmoid()
        )
        self.clsW = nn.Parameter(torch.zeros(num_targets, feat_dim))
        self.clsb = nn.Parameter(torch.zeros(num_targets))

    def encode(self, x):
        B, K, C, H, W = x.shape
        flat_x = x.view(B * K, C, H, W)
        chunk_size = 4
        feats = []
        for i in range(0, flat_x.size(0), chunk_size):
            f = self.backbone(flat_x[i:i + chunk_size])
            feats.append(f)
        feats = torch.cat(feats, dim=0)
        return feats.view(B, K, -1)

    def forward(self, x):
        feats = self.encode(x)
        h = self.norm(feats)
        g = self.gate(h)
        pooled = torch.einsum('bkn,bkf->bnf', g, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

def run_engine12_inference(fold, weight_path):
    print(f'=== Engine 12 Fold {fold} Inference ===', flush=True)
    
    val_ids = fold_studies[fold]
    val_labels = labels_df.set_index('StudyInstanceUID').loc[val_ids, TARGETS]
    
    dataset = StudyWindowsDataset(
        ids=val_ids,
        id2row=id2row,
        labels=val_labels,
        vols=vols,
        masks=masks,
        res=384, k=16, train=False
    )
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=4, pin_memory=True)
    
    # Load model
    backbone = timm.create_model('convnext_large.fb_in22k_ft_in1k_384', pretrained=False, num_classes=0)
    model = ConvNeXtMILClassifier(backbone, feat_dim=1536, num_targets=12, drop=0.2).cuda()
    
    ckpt = torch.load(weight_path, map_location='cuda')
    state_dict = ckpt['model']
    # Strip 'backbone.' prefix
    model_sd = {}
    for k, v in state_dict.items():
        if k.startswith('backbone.'):
            model_sd[k[9:]] = v
        elif k in ['clsW', 'clsb', 'img_mean', 'img_std']:
            model_sd[k] = v
    model.load_state_dict(model_sd, strict=False)
    model.eval()
    
    all_preds = []
    all_uids = []
    with torch.no_grad():
        for imgs, masks_t, y, uid in tqdm(loader, desc=f'E12 Fold {fold}'):
            imgs = imgs.cuda(non_blocking=True)
            masks_t = masks_t.cuda(non_blocking=True)
            logits = model(imgs)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs[0])
            all_uids.append(uid[0])
    
    preds_df = pd.DataFrame(all_preds, columns=TARGETS)
    preds_df.insert(0, 'StudyInstanceUID', all_uids)
    
    # Save
    out_path = f'/content/engine12_fold{fold}_oof.csv'
    preds_df.to_csv(out_path, index=False)
    print(f'[+] Saved Engine 12 Fold {fold} OOF: {out_path}')
    
    del model
    torch.cuda.empty_cache()
    return preds_df

# Run for all 4 folds
e12_oofs = {}
for fold in [1, 2, 3, 4]:
    weight_path = f'/kaggle/input/rsna-knee-engine12-fold{fold}-weights/engine12_convnext_large_fold{fold}_best.pt'
    e12_oofs[fold] = run_engine12_inference(fold, weight_path)

# Combine all folds
e12_combined = pd.concat([e12_oofs[f] for f in [1,2,3,4]], ignore_index=True)
e12_combined.to_csv('/content/engine12_all_folds_oof.csv', index=False)
print('[+] Engine 12 All Folds OOF saved')

In [ ]:
# 7. Engine 13: Swin-Base-384 Inference (All 4 Folds)
import torch, torch.nn as nn, torch.nn.functional as F
import timm
from tqdm import tqdm

class SwinMILClassifier(nn.Module):
    def __init__(self, backbone, feat_dim=1024, num_targets=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.feat_dim = feat_dim
        self.norm = nn.LayerNorm(feat_dim)
        self.att = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.Tanh(),
            nn.Dropout(drop),
            nn.Linear(256, num_targets)
        )
        self.clsW = nn.Parameter(torch.zeros(num_targets, feat_dim))
        self.clsb = nn.Parameter(torch.zeros(num_targets))

    def encode(self, x):
        B, K, C, H, W = x.shape
        flat_x = x.view(B * K, C, H, W)
        chunk_size = 4
        feats = []
        for i in range(0, flat_x.size(0), chunk_size):
            f = self.backbone(flat_x[i:i + chunk_size])
            feats.append(f)
        feats = torch.cat(feats, dim=0)
        return feats.view(B, K, -1)

    def forward(self, x):
        feats = self.encode(x)
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum('bkn,bkf->bnf', a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

def run_engine13_inference(fold, weight_path):
    print(f'=== Engine 13 Fold {fold} Inference ===', flush=True)
    
    val_ids = fold_studies[fold]
    val_labels = labels_df.set_index('StudyInstanceUID').loc[val_ids, TARGETS]
    
    dataset = StudyWindowsDataset(
        ids=val_ids,
        id2row=id2row,
        labels=val_labels,
        vols=vols,
        masks=masks,
        res=384, k=16, train=False
    )
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=4, pin_memory=True)
    
    backbone = timm.create_model('swin_base_patch4_window12_384.ms_in22k_ft_in1k', pretrained=False, num_classes=0)
    model = SwinMILClassifier(backbone, feat_dim=1024, num_targets=12, drop=0.2).cuda()
    
    ckpt = torch.load(weight_path, map_location='cuda')
    model.load_state_dict(ckpt['model'], strict=False)
    model.eval()
    
    all_preds = []
    all_uids = []
    with torch.no_grad():
        for imgs, masks_t, y, uid in tqdm(loader, desc=f'E13 Fold {fold}'):
            imgs = imgs.cuda(non_blocking=True)
            masks_t = masks_t.cuda(non_blocking=True)
            logits = model(imgs)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs[0])
            all_uids.append(uid[0])
    
    preds_df = pd.DataFrame(all_preds, columns=TARGETS)
    preds_df.insert(0, 'StudyInstanceUID', all_uids)
    
    out_path = f'/content/engine13_fold{fold}_oof.csv'
    preds_df.to_csv(out_path, index=False)
    print(f'[+] Saved Engine 13 Fold {fold} OOF: {out_path}')
    
    del model
    torch.cuda.empty_cache()
    return preds_df

# Run for all 4 folds
e13_oofs = {}
for fold in [1, 2, 3, 4]:
    weight_path = f'/kaggle/input/rsna-knee-engine13-fold{fold}-weights/engine13_swin_base_fold{fold}_best.pt'
    e13_oofs[fold] = run_engine13_inference(fold, weight_path)

# Combine all folds
e13_combined = pd.concat([e13_oofs[f] for f in [1,2,3,4]], ignore_index=True)
e13_combined.to_csv('/content/engine13_all_folds_oof.csv', index=False)
print('[+] Engine 13 All Folds OOF saved')

In [ ]:
# 8. Push OOF Predictions to Kaggle Datasets
import subprocess
import os

# Engine 12 All Folds OOF
os.makedirs('/content/e12_all_folds_upload', exist_ok=True)
subprocess.run(['cp', '/content/engine12_all_folds_oof.csv', '/content/e12_all_folds_upload/'], check=True)
for fold in [1,2,3,4]:
    subprocess.run(['cp', f'/content/engine12_fold{fold}_oof.csv', '/content/e12_all_folds_upload/'], check=True)

meta = {
    'title': 'RSNA Knee Engine 12 All Folds OOF Predictions',
    'id': 'cyruspinto/rsna-knee-engine12-all-folds-oof',
    'licenses': [{'name': 'CC0-1.0'}]
}
with open('/content/e12_all_folds_upload/dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

subprocess.run(['kaggle', 'datasets', 'version', '-p', '/content/e12_all_folds_upload', 
               '-m', 'Engine 12 All Folds (1-4) OOF Predictions', '-r', 'zip'], check=True)
print('[+] Published Engine 12 All Folds OOF to Kaggle!')

# Engine 13 All Folds OOF
os.makedirs('/content/e13_all_folds_upload', exist_ok=True)
subprocess.run(['cp', '/content/engine13_all_folds_oof.csv', '/content/e13_all_folds_upload/'], check=True)
for fold in [1,2,3,4]:
    subprocess.run(['cp', f'/content/engine13_fold{fold}_oof.csv', '/content/e13_all_folds_upload/'], check=True)

meta = {
    'title': 'RSNA Knee Engine 13 All Folds OOF Predictions',
    'id': 'cyruspinto/rsna-knee-engine13-all-folds-oof',
    'licenses': [{'name': 'CC0-1.0'}]
}
with open('/content/e13_all_folds_upload/dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

subprocess.run(['kaggle', 'datasets', 'version', '-p', '/content/e13_all_folds_upload', 
               '-m', 'Engine 13 All Folds (1-4) OOF Predictions', '-r', 'zip'], check=True)
print('[+] Published Engine 13 All Folds OOF to Kaggle!')

# Also save combined parquet for optimization script
e12_combined.to_parquet('/content/engine12_oof.parquet')
e13_combined.to_parquet('/content/engine13_oof.parquet')
print('[+] Saved parquet files for optimization')